In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# Генерація вихідних даних для Варіанта 1 (Київ)
np.random.seed(1)
city = "Київ"
monthly_temp = [-4, -3, 2, 9, 15, 19, 21, 20, 14, 8, 2, -2]
months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", 
          "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]

rows = []
for month, temp in zip(months, monthly_temp):
    heating_days = np.clip(18 - temp + np.random.normal(0, 1.5), 0, 30)
    heating_days = round(heating_days)
    cost = 0.15 * heating_days + np.random.normal(0, 1.0)
    cost = round(max(cost, 0), 1)
    rows.append({
        "місто": city, "місяць": month, "температура": temp,
        "опалювальні_дні": heating_days, "витрати_на_опалення": cost,
    })

climate = pd.DataFrame(rows)
display(climate)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(climate["температура"], climate["витрати_на_опалення"], color='royalblue', s=70, zorder=3)
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_xlabel("Середньомісячна температура, °C")
ax.set_ylabel("Витрати на опалення, умовних тис. грн")
ax.set_title("Залежність витрат на опалення від середньомісячної температури (Київ)")
plt.show()

In [ ]:
r_res = pearsonr(climate["температура"], climate["витрати_на_опалення"])
rho_res = spearmanr(climate["температура"], climate["витрати_на_опалення"])

print(f"Пірсон: r = {r_res.statistic:.3f}, p-value = {r_res.pvalue:.4e}")
print(f"Спірмен: rho = {rho_res.statistic:.3f}, p-value = {rho_res.pvalue:.4e}")

ci = r_res.confidence_interval(confidence_level=0.95)
print(f"95% довірчий інтервал для Пірсона: [{ci.low:.3f}, {ci.high:.3f}]")

In [ ]:
numeric_cols = climate[["температура", "опалювальні_дні", "витрати_на_опалення"]]

print("Матриця Пірсона:")
display(numeric_cols.corr(method="pearson"))

print("\nМатриця Спірмена:")
display(numeric_cols.corr(method="spearman"))

In [ ]:
climate_anomaly = climate.copy()
# Штучно збільшуємо витрати у грудні в 5 разів (аварія на котельні)
climate_anomaly.loc[climate_anomaly["місяць"] == "Гру", "витрати_на_опалення"] *= 5

r_anom = pearsonr(climate_anomaly["температура"], climate_anomaly["витрати_на_опалення"])
rho_anom = spearmanr(climate_anomaly["температура"], climate_anomaly["витрати_на_опалення"])

print(f"З аномалією - Пірсон: r = {r_anom.statistic:.3f}, p = {r_anom.pvalue:.4f}")
print(f"З аномалією - Спірмен: rho = {rho_anom.statistic:.3f}, p = {rho_anom.pvalue:.4f}")

Контрольні питання та теоретичні відповідіМеханіка коефіцієнта Пірсона:Коефіцієнт Пірсона ділиться на добуток стандартних відхилень обох змінних, оскільки він є коваріацією, нормованою на масштаб вимірювання обох змінних. Ділення на стандартні відхилення переводить змінні у безрозмірні стандартизовані величини (Z-оцінки). Завдяки такій нормі результат гарантовано лежить у діапазоні від -1 до 1.Чому Спірмен рахується на рангах:Використання рангів замість вихідних значень дозволяє вимірювати нелінійні монотонні зв'язки та забезпечує стійкість до викидів. Екстремальні значення перетворюються на звичайні порядкові номери, що нейтралізує їхній руйнівний вплив на результат.Приклад, коли r ≈ 0 при сильному зв'язку:Класичним прикладом є параболічна залежність (наприклад, $y = x^2$ на інтервалі від -2 до 2). Зв'язок ідеально функціональний і детермінований, проте коефіцієнт Пірсона дорівнюватиме нулю, оскільки він вимірює лише лінійну складову, а симетрія параболи компенсує зростання та спад.Чому сильна кореляція не доводить причинність:Кореляція лише констатує наявність статистичного зв'язку. Вона не виключає наявність спільного конфаундера (третього чинника), наявності зворотної причини (коли Y впливає на X) або випадкового збігу на обмеженій вибірці.